In [28]:
import pandas as pd
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler, BorderlineSMOTE, SVMSMOTE, KMeansSMOTE, SMOTENC
from imblearn.under_sampling import ClusterCentroids
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np


test_df = pd.read_csv('spacecraft_test_preprocessed.csv')
train_df = pd.read_csv('spacecraft_train_preprocessed.csv')

In [29]:
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
label_encoder = LabelEncoder()

# Fit and transform the 'race' column to convert categories to numbers
train_df['race'] = label_encoder.fit_transform(train_df['race'])


In [34]:
train_df

tournament  game_length  winner  max_collection_rate  apm   spm  \
0            0.0          848       0                 3778  316  26.3   
1            1.0          612       1                 4344  398  42.9   
2            1.0          486       0                 3095  389  35.9   
3            0.0         1078       0                 4064  449  40.2   
4            2.0         1001       1                 4484  627  37.6   
...          ...          ...     ...                  ...  ...   ...   
1653         2.0         1059       0                 3471  415  33.9   
1654         1.0          473       0                 2625  410  16.5   
1655         1.0          650       1                 3510  335  45.0   
1656         0.0          881       0                 3862  379  45.5   
1657         1.0          447       1                 2311  296  45.5   

      workers_produced  workers_killed  workers_lost  supply_block  ...  \
0                   75              18            17            75  ...   
1                   97              22            21            30  ...   
2                   80              11            17            20  ...   
3                   77              68            30            15  ...   
4                   98              24            25            25  ...   
...                ...             ...           ...           ...  ...   
1653                89              54            51            45  ...   
1654                49               3             4            20  ...   
1655                66               7             3             5  ...   
1656               100              30            37            40  ...   
1657                48              18             6            50  ...   

      avg_pac_gap  avg_collection_rate_minerals  avg_unspent_minerals  \
0            0.06                        1808.2                 241.2   
1            0.06                        1938.9                 473.6   
2            0.06                        1563.7                 232.1   
3            0.06                        2107.6                 535.6   
4            0.04                        2306.8                 259.8   
...           ...                           ...                   ...   
1653         0.07                        1583.5                 341.4   
1654         0.04                        1246.9                 150.1   
1655         0.08                        1876.5                 268.4   
1656         0.08                        1622.3                 292.6   
1657         0.07                        1064.1                 122.5   

      minerals_lost  minerals_collected  avg_collection_rate_gas  \
0             10341               25162                    533.8   
1              3075               20305                    473.9   
2              4200               13525                    226.5   
3             13650               37470                    749.7   
4             21675               37048                    826.1   
...             ...                 ...                      ...   
1653          12968               27055                    446.9   
1654           1725               10705                    332.4   
1655           8575               21220                    457.3   
1656           5535               24609                    580.9   
1657            475                8575                    269.3   

      avg_unspent_gas  gas_lost  gas_collected  race  
0               273.8      3436           6384     1  
1               236.7       625           4265     2  
2               157.6       800           1668     2  
3               401.2      5850          11682     0  
4               299.8      8975          11930     2  
...               ...       ...            ...   ...  
1653            147.0      4018           6423     2  
1654            129.8       600           2432     0  
1655            195.0      1250        

In [38]:
class_weights_dict = dict()
n, num_class = train_df.shape[0], len(train_df['race'].unique())
for label in train_df['race'].unique():
    n_l = train_df[train_df['race'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

{1: 1.0368980612883052, 2: 1.1586303284416493, 0: 0.852880658436214}

In [39]:
# Assuming train_df has the target column 'race' and features are in X_train
sample_weights = train_df['race'].map(class_weights_dict).values

# # Train the XGBoost model
# xgb_model.fit(X, y, sample_weight=sample_weights)

In [40]:
len(sample_weights)

1658

In [41]:
X = train_df.drop('race', axis=1)
y = train_df['race']

model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

f1_list = []

# split data into X and y using cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_index, test_index in kf.split(X, y):
    # print(len(train_index))
    # Oversample only on training set
    y_train = y.iloc[train_index]

    # Fit the model on the resampled dataset
    model.fit(X.iloc[train_index], y_train, sample_weight=sample_weights[train_index])

    # Predict on the test set
    y_pred = model.predict(X.iloc[test_index])

    # Calculate f1 score using weighted averaging (for multiclass)
    f1 = f1_score(y.iloc[test_index], y_pred, average='weighted')
    f1_list.append(f1)

print(f'Average F1 score for weighted: {np.mean(f1_list):.4f}')

Average F1 score for weighted: 0.8425


In [37]:
X = train_df.drop('race', axis=1)
y = train_df['race']

model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

f1_list = []

# split data into X and y using cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_index, test_index in kf.split(X, y):
    # print(len(train_index))
    # Oversample only on training set
    y_train = y.iloc[train_index]

    # Fit the model on the resampled dataset
    model.fit(X.iloc[train_index], y_train)

    # Predict on the test set
    y_pred = model.predict(X.iloc[test_index])

    # Calculate f1 score using weighted averaging (for multiclass)
    f1 = f1_score(y.iloc[test_index], y_pred, average='weighted')
    f1_list.append(f1)

print(f'Average F1 score for weighted: {np.mean(f1_list):.4f}')

Average F1 score for weighted: 0.8477


In [36]:

# Assuming your dataset has numerical features only. If it has categorical features, you can use SMOTENC with categorical indices.
# Define the oversampling methods
methods = [
    SMOTE(sampling_strategy='auto'),
    ADASYN(sampling_strategy='minority'),
    RandomOverSampler(sampling_strategy='auto'),
    BorderlineSMOTE(sampling_strategy='auto'),  # Focuses on borderline cases
    SVMSMOTE(sampling_strategy='auto'),  # Uses SVM to generate synthetic samples
    # KMeansSMOTE(sampling_strategy='auto', kmeans_estimator=3),  # Uses KMeans clustering to generate synthetic samples
    ClusterCentroids(sampling_strategy='auto')  # Under-sampling method for reducing majority class
]
# X and y should come from your data, replace train_df with your DataFrame
X = train_df.drop('race', axis=1)
y = train_df['race']

# Initialize arrays for average f1-scores across folds
for method in methods:
    # define the model
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

    f1_list = []

    # split data into X and y using cross-validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for train_index, test_index in kf.split(X, y):
        # print(len(train_index))
        # Oversample only on training set
        y_train = y.iloc[train_index]
        # print(y_train.value_counts())
        X_resampled, y_resampled = method.fit_resample(X.iloc[train_index], y.iloc[train_index])
        

        # Fit the model on the resampled dataset
        model.fit(X_resampled, y_resampled)

        # Predict on the test set
        y_pred = model.predict(X.iloc[test_index])

        # Calculate f1 score using weighted averaging (for multiclass)
        f1 = f1_score(y.iloc[test_index], y_pred, average='weighted')
        f1_list.append(f1)

    print(f'Average F1 score for {method.__class__.__name__}: {np.mean(f1_list):.4f}')


Average F1 score for SMOTE: 0.8441
Average F1 score for ADASYN: 0.8449
Average F1 score for RandomOverSampler: 0.8520
Average F1 score for BorderlineSMOTE: 0.8519


KeyboardInterrupt: 

In [26]:
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, ClusterCentroids, NearMiss

# Define the undersampling methods
methods = [
    RandomUnderSampler(sampling_strategy='auto'),  # Randomly under-samples the majority class
    TomekLinks(sampling_strategy='auto'),  # Removes Tomek links to clean boundary between classes
    ClusterCentroids(sampling_strategy='auto'),  # Reduces the majority class by using KMeans centroids
    NearMiss(sampling_strategy='auto', version=1)  # Selects majority class samples with the smallest average distance to minority class
]
# X and y should come from your data, replace train_df with your DataFrame
X = train_df.drop('race', axis=1)
y = train_df['race']

# Initialize arrays for average f1-scores across folds
for method in methods:
    # define the model
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

    f1_list = []

    # split data into X and y using cross-validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for train_index, test_index in kf.split(X, y):
        # print(len(train_index))
        # Oversample only on training set
        y_train = y.iloc[train_index]
        # print(y_train.value_counts())
        X_resampled, y_resampled = method.fit_resample(X.iloc[train_index], y.iloc[train_index])
        

        # Fit the model on the resampled dataset
        model.fit(X_resampled, y_resampled)

        # Predict on the test set
        y_pred = model.predict(X.iloc[test_index])

        # Calculate f1 score using weighted averaging (for multiclass)
        f1 = f1_score(y.iloc[test_index], y_pred, average='weighted')
        f1_list.append(f1)

    print(f'Average F1 score for {method.__class__.__name__}: {np.mean(f1_list):.4f}')


Average F1 score for RandomUnderSampler: 0.8399
Average F1 score for TomekLinks: 0.8351


Average F1 score for ClusterCentroids: 0.8307
Average F1 score for NearMiss: 0.7952
